# MMA Decision Lab: a worked decision-tree calculation

## Goal
Verify how one tactical policy's terminal branches produce expected utility. These are **mathematical teaching inputs**, not observed UFC probabilities or invented fight records. The real historical workflow is separate.

## Setup
Install the repository's locked environment. Run from the repository root or `notebooks/`. The neutral baseline is assumed to be 0.60; win utility is 1 and loss utility is 0. No source dataset is required for this arithmetic example.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.special import expit, logit

root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
from mma_decision_lab.schemas import ScenarioConfig
from mma_decision_lab.decision import POLICIES, decision_tree, expected_utilities
pd.set_option('display.precision', 6)
p0 = 0.60
chosen = POLICIES[1]  # Grappling priority; static allocation, not guaranteed fight time.


## Steps
### 1. Inspect the neutral terminal branches
Response weights are 0.25 / 0.50 / 0.25. All tactical effects are zero. Each response has a win and a loss terminal consequence.

In [2]:
neutral = ScenarioConfig()
branches = decision_tree(p0, neutral)
selected = branches[branches.policy == chosen.name]
selected[['response', 'response_probability', 'outcome', 'conditional_outcome_probability', 'utility', 'joint_probability', 'utility_contribution']]

,response,response_probability,outcome,conditional_outcome_probability,utility,joint_probability,utility_contribution
6,Lower resistance,0.25,Win,0.6,1,0.15,0.15
7,Lower resistance,0.25,Loss,0.4,0,0.10,0.00
8,Expected resistance,0.50,Win,0.6,1,0.30,0.30
9,Expected resistance,0.50,Loss,0.4,0,0.20,0.00
10,Stronger resistance,0.25,Win,0.6,1,0.15,0.15
11,Stronger resistance,0.25,Loss,0.4,0,0.10,0.00


### 2. Check the utility by hand
The win contributions are `0.25×0.60 + 0.50×0.60 + 0.25×0.60 = 0.60`. Loss contributions are zero. This is an exact expectation, not a simulation estimate.

In [3]:
manual_neutral = 0.25 * 0.60 + 0.50 * 0.60 + 0.25 * 0.60
engine_neutral = selected.utility_contribution.sum()
assert np.isclose(selected.joint_probability.sum(), 1.0)
assert np.isclose(engine_neutral, manual_neutral)
assert np.allclose(expected_utilities(p0, neutral), p0)
print(f'Hand calculation: {manual_neutral:.6f}; engine: {engine_neutral:.6f}')

Hand calculation: 0.600000; engine: 0.600000


### 3. Make the assumed tactical effect explicit
For this second **hypothetical** arithmetic example, use `s=.4, g=.2, R=.25, c=.1`. The policy allocation is fixed at standing/clinch/ground = .20/.30/.50. The response index is `clip(R + offset, 0, 1)`, with offsets −.25 / 0 / +.25. Calculate delta, conditional P(win), and branch contribution separately. These coefficients are assumptions, not paper estimates.

In [4]:
assumed = ScenarioConfig(striking_effect=.4, grappling_effect=.2, resistance=.25, counter_cost=.1)
distance_difference = chosen.allocation[0] - .5
grappling_difference = chosen.allocation[1] + chosen.allocation[2] - .5
rows = []
for label, offset, weight in zip(['Lower', 'Expected', 'Stronger'], [-.25, 0, .25], assumed.response_weights):
    response_index = np.clip(assumed.resistance + offset, 0, 1)
    delta = (.4 * distance_difference + .2 * grappling_difference * (1 - 2 * response_index)
             - .1 * abs(grappling_difference) * response_index)
    conditional = expit(logit(p0) + delta)
    rows.append({'Response': label, 'Weight': weight, 'Resistance index': response_index,
                 'Assumed delta': delta, 'Conditional P(win)': conditional,
                 'Expected utility contribution': weight * conditional})
worked = pd.DataFrame(rows)
worked

,Response,Weight,Resistance index,Assumed delta,Conditional P(win),Expected utility contribution
0,Lower,0.25,0.00,-0.0600,0.585517,0.146379
1,Expected,0.50,0.25,-0.0975,0.576388,0.288194
2,Stronger,0.25,0.50,-0.1350,0.567207,0.141802


## Checks
Sum the displayed branch contributions and compare with the production engine. The result concerns this one assumed policy/scenario model; no empirical policy recommendation follows.

In [5]:
manual_value = worked['Expected utility contribution'].sum()
engine_value = expected_utilities(p0, assumed)[1]
assert np.isclose(manual_value, engine_value, atol=1e-12)
assert 0 < engine_value < 1
print(f'Hypothetical grappling policy expected utility: {engine_value:.6f}')
print('All probability-mass, neutral-invariant and hand-calculation checks passed.')

Hypothetical grappling policy expected utility: 0.576375
All probability-mass, neutral-invariant and hand-calculation checks passed.


## Next steps
Run `python scripts/run_example_analysis.py` from the repository root for the real historical example, or use `--illustrative` to deliberately enable hypothetical nonneutral assumptions. Read `docs/METHODOLOGY.md` and `docs/MODEL_CARD.md` before interpreting the research outputs. The historical model is exploratory at the default cutoff, and public observations do not identify tactical causal effects.

The tutorial intentionally uses small branch tables rather than charts: its purpose is to audit a single expected-utility calculation. The full application and standalone research report contain the uncertainty and sensitivity visualizations.